# Day 6: Advanced LangGraph  
## Human Review, Error Recovery and Multi-Agent Flow  
### Insurance Claim Settlement Workflow

**Audience:** Beginner to Intermediate UiPath Developers, RPA Developers, Automation Engineers, AI Automation Teams  
**Duration:** 4 Hours  
**Domain:** Insurance Claim Settlement, Medical Claims, Motor Claims, Travel Claims, Human Review, Retry and Escalation

---

## Learning Outcomes

By the end of this notebook, participants will be able to:

- Understand LangChain agentic AI use cases for insurance.
- Understand LangChain vs LangGraph in practical automation terms.
- Build LangChain examples using tools, memory and LLM model.
- Use 4–5 insurance-focused LangChain tools/agents.
- Use LLM model inside LangGraph nodes.
- Design LangGraph human-in-the-loop review checkpoints.
- Build approval, rejection, escalation and rework routing.
- Add retry and error recovery strategies.
- Build medical, motor and travel claim routing.
- Build a multi-agent claim settlement workflow with human review.

# 0. How to Use This Notebook

This notebook is designed as a practical lab workbook.

Recommended flow:

1. Read the concept section.
2. Run the setup cells.
3. Execute each lab step by step.
4. Modify insurance claim examples.
5. Complete mini assignments.
6. Review solution examples at the end.
7. Build the final Multi-Agent Claim Settlement Workflow.

Environment variables are assumed to be configured:

```text
OPENROUTER_API_KEY=your_api_key_here
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
OPENROUTER_MODEL=openai/gpt-4.1-mini
```

Most labs first use deterministic Python logic, then add LLM calls to make the workflow more realistic.

# 1. Day 6 Agenda

| Time | Topic |
|---|---|
| 25 min | LangChain vs LangGraph |
| 35 min | LangChain with tools, memory and LLM |
| 35 min | Insurance-domain LangChain agents |
| 35 min | LangGraph with LLM nodes |
| 35 min | Human review checkpoint |
| 35 min | Approval, rejection, escalation and rework routing |
| 30 min | Retry and error recovery |
| 35 min | Medical, motor and travel routing |
| 30 min | Final multi-agent claim settlement assignment |

---

## Trainer Humour

A normal bot says:  
“Missing field. Process failed.”

A mature AI workflow says:  
“Missing field. Routed to rework. Logged for audit. Human can review. Nobody panics.”

# 2. Install Required Libraries

Required packages:

- openai
- python-dotenv
- langchain
- langchain-openai
- langgraph
- pydantic
- pandas

Install only if missing.

In [ ]:
# Uncomment only if packages are missing.
# !pip install openai python-dotenv langchain langchain-openai langgraph pydantic pandas

# 3. API Setup and Reusable LLM Call

We will use an OpenAI-compatible client through OpenRouter.

The same `call_llm()` function will be used inside:

- LangChain examples
- LangGraph nodes
- Human review summaries
- Escalation messages
- Final settlement report generation

In [1]:
import os
import json
import time
import random
from typing import TypedDict, Dict, Any, List, Optional
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

print("Base URL:", OPENROUTER_BASE_URL)
print("Model:", OPENROUTER_MODEL)
print("API Key Loaded:", bool(OPENROUTER_API_KEY))

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY
)

def call_llm(prompt, system_message="You are an insurance automation assistant.", temperature=0.2):
    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

Base URL: https://openrouter.ai/api/v1
Model: openai/gpt-4.1-mini
API Key Loaded: True


# 4. LangChain vs LangGraph

## LangChain

LangChain is useful when you need:

- Prompt templates
- Chains
- Tools
- Memory
- Simple agents
- RAG-based assistants
- LLM application building blocks

## LangGraph

LangGraph is useful when you need:

- Stateful workflows
- Multi-step agent orchestration
- Conditional routing
- Human review checkpoints
- Retry and recovery paths
- Long-running business processes
- Multi-agent collaboration

## Insurance Example

| Requirement | Better Fit |
|---|---|
| Policy FAQ assistant | LangChain |
| Claim settlement workflow | LangGraph |
| Tool-based claim lookup | LangChain |
| Human approval and rework routing | LangGraph |
| One prompt output | LangChain |
| Multi-agent claim process | LangGraph |

# 5. Sample Insurance Data

We will use medical, motor and travel claims.

In [2]:
medical_claim = {
    "claim_id": "MED1001",
    "policy_number": "POL1001",
    "customer_name": "Rajesh Sharma",
    "claim_type": "Medical",
    "claim_amount": 185000,
    "description": "Hospitalization for dengue fever. Customer submitted hospital bill and discharge summary.",
    "submitted_documents": ["Hospital Bill", "Discharge Summary", "Doctor Prescription"],
    "priority": "Normal"
}

high_value_medical_claim = {
    "claim_id": "MED9001",
    "policy_number": "POL1001",
    "customer_name": "Rajesh Sharma",
    "claim_type": "Medical",
    "claim_amount": 850000,
    "description": "Major surgery claim with high hospital bill and ICU charges.",
    "submitted_documents": ["Hospital Bill", "Discharge Summary", "Doctor Prescription", "Claim Form"],
    "priority": "High"
}

motor_claim = {
    "claim_id": "MOT2001",
    "policy_number": "VEH2045",
    "customer_name": "Meena Joshi",
    "claim_type": "Motor",
    "claim_amount": 240000,
    "description": "Vehicle accident at midnight. Repair estimate is high. Photos are unclear.",
    "submitted_documents": ["Repair Estimate", "Vehicle Photos"],
    "priority": "Normal"
}

travel_claim = {
    "claim_id": "TRV3001",
    "policy_number": "TRV3001",
    "customer_name": "Amit Verma",
    "claim_type": "Travel",
    "claim_amount": 18000,
    "description": "Baggage delayed for 48 hours during international travel.",
    "submitted_documents": ["Boarding Pass", "Airline Delay Certificate"],
    "priority": "Normal"
}

print("Sample claims loaded.")

Sample claims loaded.


In [3]:
policy_database = {
    "POL1001": {
        "policy_type": "Medical",
        "policy_status": "Active",
        "premium_status": "Paid",
        "coverage_amount": 1000000,
        "customer_name": "Rajesh Sharma"
    },
    "VEH2045": {
        "policy_type": "Motor",
        "policy_status": "Active",
        "premium_status": "Paid",
        "coverage_amount": 300000,
        "customer_name": "Meena Joshi"
    },
    "TRV3001": {
        "policy_type": "Travel",
        "policy_status": "Active",
        "premium_status": "Paid",
        "coverage_amount": 100000,
        "customer_name": "Amit Verma"
    }
}

required_documents = {
    "Medical": ["Hospital Bill", "Discharge Summary", "Doctor Prescription", "Claim Form"],
    "Motor": ["Repair Estimate", "Vehicle Photos", "Police Report", "Driving License", "RC Copy"],
    "Travel": ["Boarding Pass", "Airline Delay Certificate", "Baggage Delay Report", "Claim Form"]
}

print("Mock enterprise databases loaded.")

Mock enterprise databases loaded.


# 6. LangChain with Memory, Tools and LLM — Complete Example

## Objective

Create a simple insurance assistant that can:

- Remember conversation context
- Use tools
- Answer policy and claim questions
- Use LLM to produce customer-friendly responses

We will build the logic step by step.

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0.2
)

conversation_memory = []

print("LangChain LLM and memory list initialized.")

LangChain LLM and memory list initialized.


## 6.1 Create Insurance Tools

Tools are Python functions that provide factual data.

We will create 5 tools:

1. Policy Lookup Tool  
2. Claim Document Checklist Tool  
3. Claim Amount Severity Tool  
4. Claim Type Routing Tool  
5. Settlement Eligibility Tool

In [5]:
def policy_lookup_tool(policy_number):
    return policy_database.get(policy_number, {"error": "Policy not found"})

def document_checklist_tool(claim_type):
    return required_documents.get(claim_type, [])

def claim_amount_severity_tool(claim_amount):
    if claim_amount < 50000:
        return "Low"
    elif claim_amount <= 300000:
        return "Medium"
    else:
        return "High"

def claim_type_routing_tool(claim_type):
    routing = {
        "Medical": "Medical Claims Team",
        "Motor": "Motor Claims Team",
        "Travel": "Travel Claims Team"
    }
    return routing.get(claim_type, "General Claims Team")

def settlement_eligibility_tool(claim):
    policy = policy_lookup_tool(claim["policy_number"])

    if "error" in policy:
        return {"eligible": False, "reason": "Policy not found"}

    if policy["policy_status"] != "Active":
        return {"eligible": False, "reason": "Policy inactive"}

    if policy["premium_status"] != "Paid":
        return {"eligible": False, "reason": "Premium pending"}

    if claim["claim_amount"] > policy["coverage_amount"]:
        return {"eligible": False, "reason": "Claim amount exceeds coverage"}

    return {"eligible": True, "reason": "Policy valid and claim within coverage"}

print(policy_lookup_tool("POL1001"))
print(document_checklist_tool("Medical"))
print(claim_amount_severity_tool(185000))
print(claim_type_routing_tool("Motor"))
print(settlement_eligibility_tool(medical_claim))

{'policy_type': 'Medical', 'policy_status': 'Active', 'premium_status': 'Paid', 'coverage_amount': 1000000, 'customer_name': 'Rajesh Sharma'}
['Hospital Bill', 'Discharge Summary', 'Doctor Prescription', 'Claim Form']
Medium
Motor Claims Team
{'eligible': True, 'reason': 'Policy valid and claim within coverage'}


## 6.2 Create LangChain Assistant with Memory

This example uses a simple memory list.

Each interaction is added to memory so the next response has context.

In [15]:
assistant_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an insurance helpdesk assistant. Use professional and factual language."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{question}")
])

assistant_chain = assistant_prompt | llm | StrOutputParser()

def langchain_memory_assistant(question):
    global conversation_memory

    response = assistant_chain.invoke({
        "history": conversation_memory,
        "question": question
    })

    conversation_memory.append(HumanMessage(content=question))
    conversation_memory.append(AIMessage(content=response))

    return response

# Uncomment after API key setup.
#print(langchain_memory_assistant("My claim MED1001 is for dengue hospitalization. What documents may be required?"))
print(langchain_memory_assistant("Can you remind me what claim I asked about?"))

You asked about claim MED1001, which is related to dengue hospitalization.


## Mini Assignment 1

Enhance the assistant to call `document_checklist_tool()` before answering document-related questions.

Expected behavior:

- If user asks about documents for Medical claim, tool provides required list.
- LLM converts tool output into customer-friendly answer.

# 7. LangChain Multi-Agent Use Case for Insurance

We will simulate 5 LangChain-style agents as functions:

1. Intake Agent  
2. Policy Agent  
3. Document Agent  
4. Settlement Agent  
5. Customer Communication Agent  

Each agent uses either a tool, LLM or both.

In [9]:
def lc_intake_agent(claim):
    prompt = f"""
You are a claim intake agent.

Create a concise intake summary.

Claim:
{json.dumps(claim, indent=2)}
"""
    return call_llm(prompt)

def lc_policy_agent(claim):
    policy = policy_lookup_tool(claim["policy_number"])
    prompt = f"""
You are a policy verification agent.

Explain policy validation result in concise business language.

Claim:
{json.dumps(claim, indent=2)}

Policy:
{json.dumps(policy, indent=2)}
"""
    return call_llm(prompt)

def lc_document_agent(claim):
    required = document_checklist_tool(claim["claim_type"])
    submitted = claim["submitted_documents"]
    missing = [doc for doc in required if doc not in submitted]
    return {
        "required_documents": required,
        "submitted_documents": submitted,
        "missing_documents": missing,
        "document_status": "Complete" if not missing else "Missing Documents"
    }

def lc_settlement_agent(claim):
    return settlement_eligibility_tool(claim)

def lc_customer_communication_agent(claim, settlement_result, document_result):
    prompt = f"""
You are an insurance customer communication assistant.

Draft a professional claim status message.

Rules:
- Do not promise approval unless settlement_result says eligible.
- Mention missing documents if any.
- Keep it short.

Claim:
{json.dumps(claim, indent=2)}

Settlement Result:
{json.dumps(settlement_result, indent=2)}

Document Result:
{json.dumps(document_result, indent=2)}
"""
    return call_llm(prompt)

In [12]:
# Complete LangChain-style multi-agent execution

claim = medical_claim

# Uncomment after API key setup.
intake = lc_intake_agent(claim)
policy_review = lc_policy_agent(claim)

document_result = lc_document_agent(claim)
settlement_result = lc_settlement_agent(claim)

# Uncomment after API key setup.
customer_message = lc_customer_communication_agent(claim, settlement_result, document_result)

print("Document Result:")
print(json.dumps(document_result, indent=2))

print("\nSettlement Result:")
print(json.dumps(settlement_result, indent=2))

# print("\nIntake:")
print(intake)
# print("\nPolicy Review:")
print(policy_review)
# print("\nCustomer Message:")
print(customer_message)

Document Result:
{
  "required_documents": [
    "Hospital Bill",
    "Discharge Summary",
    "Doctor Prescription",
    "Claim Form"
  ],
  "submitted_documents": [
    "Hospital Bill",
    "Discharge Summary",
    "Doctor Prescription"
  ],
  "missing_documents": [
    "Claim Form"
  ],
  "document_status": "Missing Documents"
}

Settlement Result:
{
  "eligible": true,
  "reason": "Policy valid and claim within coverage"
}
Claim Intake Summary:

Claim ID: MED1001  
Policy Number: POL1001  
Customer Name: Rajesh Sharma  
Claim Type: Medical  
Claim Amount: ₹185,000  
Description: Hospitalization for dengue fever. Submitted documents include hospital bill, discharge summary, and doctor prescription.  
Priority: Normal
The claim for hospitalization due to dengue fever submitted by Rajesh Sharma under policy POL1001 is valid. The policy is active with premiums paid, covers medical claims, and has a coverage limit of 1,000,000, which exceeds the claimed amount of 185,000. All required d

## Mini Assignment 2

Run the LangChain-style multi-agent flow for:

1. Medical claim  
2. Motor claim  
3. Travel claim  

Compare:

- Missing documents
- Settlement eligibility
- Customer message

# 8. LangGraph State for Advanced Claim Settlement

Now we move from LangChain-style functions to LangGraph workflow orchestration.

LangGraph is better for:

- State updates
- Conditional routing
- Human review
- Retry paths
- Rework routing
- Multi-agent collaboration

In [ ]:
from langgraph.graph import StateGraph, START, END

class SettlementState(TypedDict, total=False):
    claim: Dict[str, Any]
    intake_summary: str
    policy_details: Dict[str, Any]
    policy_status: str
    required_documents: List[str]
    missing_documents: List[str]
    document_status: str
    settlement_eligibility: Dict[str, Any]
    risk_level: str
    risk_indicators: List[str]
    human_review_required: bool
    human_decision: str
    approval_status: str
    routing_key: str
    final_message: str
    retry_count: int
    errors: List[str]
    logs: List[str]

# 9. Use LLM Model in LangGraph Nodes

Each LangGraph node is just a Python function.

A node can:

- Use deterministic business rules
- Use tools
- Call LLM
- Update state
- Add logs
- Route to next node

In [ ]:
def graph_intake_agent(state: SettlementState) -> SettlementState:
    claim = state["claim"]

    prompt = f"""
You are a claim intake agent.

Create a concise claim intake summary.

Include:
- claim id
- customer name
- claim type
- claim amount
- short incident summary

Claim:
{json.dumps(claim, indent=2)}
"""
    summary = call_llm(prompt)

    logs = state.get("logs", [])
    logs.append("Intake Agent completed.")

    return {
        **state,
        "intake_summary": summary,
        "logs": logs
    }

def graph_policy_agent(state: SettlementState) -> SettlementState:
    claim = state["claim"]
    policy = policy_lookup_tool(claim["policy_number"])

    if "error" in policy:
        policy_status = "Policy Not Found"
    elif policy["policy_status"] != "Active":
        policy_status = "Policy Inactive"
    elif policy["premium_status"] != "Paid":
        policy_status = "Premium Pending"
    elif claim["claim_amount"] > policy["coverage_amount"]:
        policy_status = "Claim Exceeds Coverage"
    else:
        policy_status = "Policy Valid"

    logs = state.get("logs", [])
    logs.append("Policy Agent completed.")

    return {
        **state,
        "policy_details": policy,
        "policy_status": policy_status,
        "logs": logs
    }

def graph_document_agent(state: SettlementState) -> SettlementState:
    claim = state["claim"]
    required = document_checklist_tool(claim["claim_type"])
    submitted = claim["submitted_documents"]
    missing = [doc for doc in required if doc not in submitted]
    status = "Complete" if not missing else "Missing Documents"

    logs = state.get("logs", [])
    logs.append("Document Agent completed.")

    return {
        **state,
        "required_documents": required,
        "missing_documents": missing,
        "document_status": status,
        "logs": logs
    }

In [ ]:
def graph_risk_agent(state: SettlementState) -> SettlementState:
    claim = state["claim"]
    indicators = []
    description = claim.get("description", "").lower()

    if claim["claim_amount"] > 500000:
        indicators.append("High claim amount")
    if "midnight" in description:
        indicators.append("Incident occurred at midnight")
    if "unclear" in description:
        indicators.append("Unclear evidence")
    if state.get("document_status") == "Missing Documents":
        indicators.append("Required documents missing")
    if state.get("policy_status") != "Policy Valid":
        indicators.append("Policy exception present")

    if len(indicators) >= 3:
        risk_level = "High"
    elif len(indicators) >= 1:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    logs = state.get("logs", [])
    logs.append("Risk Agent completed.")

    return {
        **state,
        "risk_indicators": indicators,
        "risk_level": risk_level,
        "logs": logs
    }

def graph_settlement_agent(state: SettlementState) -> SettlementState:
    claim = state["claim"]
    eligibility = settlement_eligibility_tool(claim)

    logs = state.get("logs", [])
    logs.append("Settlement Agent completed.")

    return {
        **state,
        "settlement_eligibility": eligibility,
        "logs": logs
    }

# 10. Human-in-the-Loop Review Pattern

Human review is required when:

- Claim amount is high
- Risk level is High
- Policy exception exists
- Claim exceeds coverage
- Missing critical documents
- Manual business approval is required

In training notebooks, we simulate human decision using a state field:

```python
human_decision = "approve" / "reject" / "rework" / "escalate"
```

In real enterprise implementation, this can connect to:

- UiPath Action Center
- Email approval
- ServiceNow
- CRM task
- Claims manager dashboard

In [ ]:
def human_review_checkpoint(state: SettlementState) -> SettlementState:
    claim = state["claim"]

    high_value = claim["claim_amount"] > 500000
    high_risk = state.get("risk_level") == "High"
    policy_exception = state.get("policy_status") != "Policy Valid"

    human_review_required = high_value or high_risk or policy_exception

    logs = state.get("logs", [])
    logs.append("Human Review Checkpoint completed.")

    return {
        **state,
        "human_review_required": human_review_required,
        "logs": logs
    }

def simulated_human_review_node(state: SettlementState) -> SettlementState:
    claim = state["claim"]

    # Training simulation rule
    if state.get("document_status") == "Missing Documents":
        decision = "rework"
    elif state.get("risk_level") == "High":
        decision = "escalate"
    elif state.get("policy_status") != "Policy Valid":
        decision = "reject"
    elif claim["claim_amount"] > 500000:
        decision = "approve"
    else:
        decision = "approve"

    logs = state.get("logs", [])
    logs.append(f"Simulated Human Review completed with decision: {decision}")

    return {
        **state,
        "human_decision": decision,
        "logs": logs
    }

## Mini Assignment 3

Modify `simulated_human_review_node()` rules:

- Medical claim above INR 7,00,000 should be escalated.
- Motor claim with missing police report should be sent to rework.
- Travel claim below INR 25,000 with valid documents can be approved.

# 11. Approval, Rejection, Escalation and Rework Routing

We will create final route nodes:

- approval_node
- rejection_node
- escalation_node
- rework_node
- auto_approval_node

In [ ]:
def route_after_review(state: SettlementState) -> str:
    if state.get("human_review_required"):
        return "human_review"

    if state.get("document_status") == "Missing Documents":
        return "rework"

    if state.get("policy_status") != "Policy Valid":
        return "reject"

    if state.get("risk_level") == "High":
        return "escalate"

    return "auto_approve"

def route_human_decision(state: SettlementState) -> str:
    decision = state.get("human_decision", "escalate")

    if decision == "approve":
        return "approve"
    if decision == "reject":
        return "reject"
    if decision == "rework":
        return "rework"
    if decision == "escalate":
        return "escalate"

    return "escalate"

In [ ]:
def approval_node(state: SettlementState) -> SettlementState:
    logs = state.get("logs", [])
    logs.append("Approval Node completed.")
    return {
        **state,
        "approval_status": "Approved",
        "final_message": "Claim approved for settlement processing.",
        "logs": logs
    }

def auto_approval_node(state: SettlementState) -> SettlementState:
    logs = state.get("logs", [])
    logs.append("Auto Approval Node completed.")
    return {
        **state,
        "approval_status": "Auto Approved",
        "final_message": "Claim auto-approved based on policy, document and risk checks.",
        "logs": logs
    }

def rejection_node(state: SettlementState) -> SettlementState:
    logs = state.get("logs", [])
    logs.append("Rejection Node completed.")
    return {
        **state,
        "approval_status": "Rejected",
        "final_message": f"Claim rejected or blocked. Reason: {state.get('policy_status')}",
        "logs": logs
    }

def escalation_node(state: SettlementState) -> SettlementState:
    logs = state.get("logs", [])
    logs.append("Escalation Node completed.")
    return {
        **state,
        "approval_status": "Escalated",
        "final_message": f"Claim escalated for senior review. Risk: {state.get('risk_level')}",
        "logs": logs
    }

def rework_node(state: SettlementState) -> SettlementState:
    logs = state.get("logs", [])
    logs.append("Rework Node completed.")
    return {
        **state,
        "approval_status": "Rework Required",
        "final_message": f"Claim requires rework. Missing documents: {state.get('missing_documents')}",
        "logs": logs
    }

# 12. Retry and Error Recovery Strategy

## Why retry?

LLM or API calls may fail due to:

- Temporary network issue
- Rate limit
- Timeout
- Provider outage
- Invalid response

## Strategy

- Try up to 3 times
- Wait between attempts
- Log error
- Route to fallback or manual review if all attempts fail

In [ ]:
def call_llm_with_retry(prompt, max_retries=3, delay_seconds=2):
    errors = []

    for attempt in range(1, max_retries + 1):
        try:
            result = call_llm(prompt)
            return result, errors
        except Exception as error:
            errors.append(f"Attempt {attempt} failed: {str(error)}")
            time.sleep(delay_seconds)

    return None, errors

def safe_llm_summary_node(state: SettlementState) -> SettlementState:
    claim = state["claim"]

    prompt = f"""
Create a settlement summary for this insurance claim:

{json.dumps(claim, indent=2)}
"""

    result, errors = call_llm_with_retry(prompt)

    logs = state.get("logs", [])
    existing_errors = state.get("errors", [])

    if result is None:
        logs.append("Safe LLM Summary failed after retries.")
        existing_errors.extend(errors)
        return {
            **state,
            "final_message": "AI summary failed. Route to manual review.",
            "errors": existing_errors,
            "logs": logs
        }

    logs.append("Safe LLM Summary completed.")
    return {
        **state,
        "settlement_summary": result,
        "errors": existing_errors,
        "logs": logs
    }

## Mini Assignment 4

Modify `call_llm_with_retry()` to use exponential backoff:

```python
time.sleep(2 ** attempt)
```

Also add fallback message:

```text
AI service temporarily unavailable. Manual review required.
```

# 13. Build Advanced LangGraph Workflow

Workflow design:

```text
START
  ↓
Intake Agent
  ↓
Policy Agent
  ↓
Document Agent
  ↓
Risk Agent
  ↓
Settlement Agent
  ↓
Human Review Checkpoint
      ├── Human Review Node
      │       ├── Approve
      │       ├── Reject
      │       ├── Rework
      │       └── Escalate
      ├── Auto Approve
      ├── Rework
      ├── Reject
      └── Escalate
```

In [ ]:
advanced_graph = StateGraph(SettlementState)

advanced_graph.add_node("intake", graph_intake_agent)
advanced_graph.add_node("policy", graph_policy_agent)
advanced_graph.add_node("document", graph_document_agent)
advanced_graph.add_node("risk", graph_risk_agent)
advanced_graph.add_node("settlement", graph_settlement_agent)
advanced_graph.add_node("human_checkpoint", human_review_checkpoint)
advanced_graph.add_node("human_review", simulated_human_review_node)

advanced_graph.add_node("approve", approval_node)
advanced_graph.add_node("auto_approve", auto_approval_node)
advanced_graph.add_node("reject", rejection_node)
advanced_graph.add_node("rework", rework_node)
advanced_graph.add_node("escalate", escalation_node)

advanced_graph.add_edge(START, "intake")
advanced_graph.add_edge("intake", "policy")
advanced_graph.add_edge("policy", "document")
advanced_graph.add_edge("document", "risk")
advanced_graph.add_edge("risk", "settlement")
advanced_graph.add_edge("settlement", "human_checkpoint")

advanced_graph.add_conditional_edges(
    "human_checkpoint",
    route_after_review,
    {
        "human_review": "human_review",
        "auto_approve": "auto_approve",
        "rework": "rework",
        "reject": "reject",
        "escalate": "escalate"
    }
)

advanced_graph.add_conditional_edges(
    "human_review",
    route_human_decision,
    {
        "approve": "approve",
        "reject": "reject",
        "rework": "rework",
        "escalate": "escalate"
    }
)

advanced_graph.add_edge("approve", END)
advanced_graph.add_edge("auto_approve", END)
advanced_graph.add_edge("reject", END)
advanced_graph.add_edge("rework", END)
advanced_graph.add_edge("escalate", END)

claim_settlement_app = advanced_graph.compile()

print("Advanced LangGraph claim settlement workflow compiled.")

# 14. Run Workflow: Normal Medical Claim

Expected behavior:

- Policy valid
- Some document may be missing
- Risk usually low or medium
- Route depends on missing documents and claim value

In [ ]:
initial_state = {
    "claim": medical_claim,
    "logs": [],
    "errors": []
}

final_state = claim_settlement_app.invoke(initial_state)

print("Approval Status:", final_state.get("approval_status"))
print("Human Review Required:", final_state.get("human_review_required"))
print("Human Decision:", final_state.get("human_decision"))
print("Final Message:", final_state.get("final_message"))
print("Missing Documents:", final_state.get("missing_documents"))
print("Risk Level:", final_state.get("risk_level"))
print("Logs:")
for log in final_state.get("logs", []):
    print("-", log)

# 15. Run Workflow: High-Value Medical Claim

Expected behavior:

- High-value claim triggers human review.
- Simulated human review may approve or escalate based on rules.

In [ ]:
initial_state = {
    "claim": high_value_medical_claim,
    "logs": [],
    "errors": []
}

final_state = claim_settlement_app.invoke(initial_state)

print("Approval Status:", final_state.get("approval_status"))
print("Human Review Required:", final_state.get("human_review_required"))
print("Human Decision:", final_state.get("human_decision"))
print("Final Message:", final_state.get("final_message"))
print("Risk Level:", final_state.get("risk_level"))
print("Risk Indicators:", final_state.get("risk_indicators"))

# 16. Run Workflow: Motor Claim

Expected behavior:

- Motor claim may have missing documents.
- If police report or license is missing, route to rework.

In [ ]:
initial_state = {
    "claim": motor_claim,
    "logs": [],
    "errors": []
}

final_state = claim_settlement_app.invoke(initial_state)

print("Approval Status:", final_state.get("approval_status"))
print("Human Review Required:", final_state.get("human_review_required"))
print("Human Decision:", final_state.get("human_decision"))
print("Final Message:", final_state.get("final_message"))
print("Missing Documents:", final_state.get("missing_documents"))
print("Risk Level:", final_state.get("risk_level"))

# 17. Run Workflow: Travel Claim

Expected behavior:

- Travel claim is usually lower value.
- Missing baggage delay report or claim form may route to rework.

In [ ]:
initial_state = {
    "claim": travel_claim,
    "logs": [],
    "errors": []
}

final_state = claim_settlement_app.invoke(initial_state)

print("Approval Status:", final_state.get("approval_status"))
print("Human Review Required:", final_state.get("human_review_required"))
print("Human Decision:", final_state.get("human_decision"))
print("Final Message:", final_state.get("final_message"))
print("Missing Documents:", final_state.get("missing_documents"))
print("Risk Level:", final_state.get("risk_level"))

# 18. Example 1: Medical Claim Routing

Use this example to practice medical claim rework and approval.

In [ ]:
medical_claim_complete = {
    "claim_id": "MED1002",
    "policy_number": "POL1001",
    "customer_name": "Rajesh Sharma",
    "claim_type": "Medical",
    "claim_amount": 95000,
    "description": "Hospitalization for viral infection. All required documents are submitted.",
    "submitted_documents": ["Hospital Bill", "Discharge Summary", "Doctor Prescription", "Claim Form"],
    "priority": "Normal"
}

final_state = claim_settlement_app.invoke({
    "claim": medical_claim_complete,
    "logs": [],
    "errors": []
})

print("Approval Status:", final_state.get("approval_status"))
print("Final Message:", final_state.get("final_message"))
print("Human Review Required:", final_state.get("human_review_required"))

## Practice Assignment 1

Create a medical claim of INR 7,50,000 with all documents.

Expected:

- Human review required
- Decision based on simulated human rules
- Final route should not be automatic

# 19. Example 2: Motor Claim Escalation

Motor claim with unclear evidence and missing police report.

In [ ]:
motor_claim_escalation = {
    "claim_id": "MOT2002",
    "policy_number": "VEH2045",
    "customer_name": "Meena Joshi",
    "claim_type": "Motor",
    "claim_amount": 295000,
    "description": "Accident occurred at midnight. Photos are unclear. Repair estimate appears unusually high.",
    "submitted_documents": ["Repair Estimate"],
    "priority": "High"
}

final_state = claim_settlement_app.invoke({
    "claim": motor_claim_escalation,
    "logs": [],
    "errors": []
})

print("Approval Status:", final_state.get("approval_status"))
print("Human Decision:", final_state.get("human_decision"))
print("Final Message:", final_state.get("final_message"))
print("Risk Indicators:", final_state.get("risk_indicators"))
print("Missing Documents:", final_state.get("missing_documents"))

## Practice Assignment 2

Modify the motor claim by adding:

- Vehicle Photos
- Police Report
- Driving License
- RC Copy

Observe how routing changes.

# 20. Example 3: Travel Claim Auto Approval Scenario

Travel claim with valid policy and complete documents.

In [ ]:
travel_claim_complete = {
    "claim_id": "TRV3002",
    "policy_number": "TRV3001",
    "customer_name": "Amit Verma",
    "claim_type": "Travel",
    "claim_amount": 22000,
    "description": "Baggage delayed for 24 hours during international travel.",
    "submitted_documents": ["Boarding Pass", "Airline Delay Certificate", "Baggage Delay Report", "Claim Form"],
    "priority": "Normal"
}

final_state = claim_settlement_app.invoke({
    "claim": travel_claim_complete,
    "logs": [],
    "errors": []
})

print("Approval Status:", final_state.get("approval_status"))
print("Final Message:", final_state.get("final_message"))
print("Human Review Required:", final_state.get("human_review_required"))
print("Risk Level:", final_state.get("risk_level"))

## Practice Assignment 3

Increase travel claim amount to INR 1,20,000.

Expected:

- Claim exceeds coverage
- Policy or settlement exception
- Human review or rejection path

# 21. Example 4: Missing Document Escalation Flow

This example focuses on rework routing.

In [ ]:
missing_document_claim = {
    "claim_id": "MED1003",
    "policy_number": "POL1001",
    "customer_name": "Rajesh Sharma",
    "claim_type": "Medical",
    "claim_amount": 45000,
    "description": "Small hospitalization claim. Only hospital bill submitted.",
    "submitted_documents": ["Hospital Bill"],
    "priority": "Normal"
}

final_state = claim_settlement_app.invoke({
    "claim": missing_document_claim,
    "logs": [],
    "errors": []
})

print("Approval Status:", final_state.get("approval_status"))
print("Final Message:", final_state.get("final_message"))
print("Missing Documents:", final_state.get("missing_documents"))

## Practice Assignment 4

Add missing documents one by one and rerun workflow.

Observe how document completeness changes routing.

# 22. Example 5: Policy Exception Flow

This example uses a claim with invalid policy number.

In [ ]:
invalid_policy_claim = {
    "claim_id": "MED1004",
    "policy_number": "UNKNOWN999",
    "customer_name": "Unknown Customer",
    "claim_type": "Medical",
    "claim_amount": 75000,
    "description": "Customer submitted claim but policy number is not found.",
    "submitted_documents": ["Hospital Bill", "Discharge Summary", "Doctor Prescription", "Claim Form"],
    "priority": "Normal"
}

final_state = claim_settlement_app.invoke({
    "claim": invalid_policy_claim,
    "logs": [],
    "errors": []
})

print("Approval Status:", final_state.get("approval_status"))
print("Policy Status:", final_state.get("policy_status"))
print("Final Message:", final_state.get("final_message"))
print("Human Review Required:", final_state.get("human_review_required"))

## Practice Assignment 5

Create one claim where policy exists but premium is pending.

Hint:

- Add a new policy to `policy_database`
- Set premium_status = "Pending"
- Rerun workflow

# 23. Example 6: Batch Processing and Audit Report

Enterprise claims usually arrive as a batch from UiPath queue or claim system.

In [ ]:
claims_batch = [
    medical_claim,
    high_value_medical_claim,
    motor_claim,
    travel_claim,
    medical_claim_complete,
    motor_claim_escalation,
    travel_claim_complete,
    missing_document_claim,
    invalid_policy_claim
]

batch_results = []

for claim in claims_batch:
    final_state = claim_settlement_app.invoke({
        "claim": claim,
        "logs": [],
        "errors": []
    })

    batch_results.append({
        "claim_id": claim["claim_id"],
        "claim_type": claim["claim_type"],
        "claim_amount": claim["claim_amount"],
        "policy_status": final_state.get("policy_status"),
        "document_status": final_state.get("document_status"),
        "risk_level": final_state.get("risk_level"),
        "human_review_required": final_state.get("human_review_required"),
        "approval_status": final_state.get("approval_status"),
        "final_message": final_state.get("final_message")
    })

for result in batch_results:
    print(json.dumps(result, indent=2))
    print("-" * 80)

In [ ]:
# Optional audit report export

import pandas as pd

audit_df = pd.DataFrame(batch_results)
audit_df.to_csv("multi_agent_claim_settlement_audit_report.csv", index=False)

audit_df

## Practice Assignment 6

Add 3 more claims to `claims_batch`:

1. Property claim  
2. Life insurance claim  
3. Travel medical emergency claim  

Create routing logic or policy data as needed.

# 24. Final Assignment: Multi-Agent Claim Settlement Workflow with Human Review

## Assignment Objective

Build a Multi-Agent Claim Settlement Workflow with human review.

## Required Agents

| Agent | Responsibility |
|---|---|
| Intake Agent | Summarize claim |
| Policy Agent | Validate policy and coverage |
| Document Agent | Check missing documents |
| Risk Agent | Identify risk indicators |
| Settlement Agent | Check settlement eligibility |
| Human Review Agent | Simulate approval, rejection, rework, escalation |
| Final Routing Agent | Route to approval, rejection, escalation or rework |

## Required Routes

| Route | Condition |
|---|---|
| Auto Approval | Low risk, valid policy, complete documents |
| Human Review | High value or high risk |
| Rework | Missing documents |
| Rejection | Invalid policy or blocked claim |
| Escalation | High risk or senior review needed |

## Deliverables

Participants must submit:

1. Notebook with completed workflow
2. At least 5 test claims
3. Human review logic
4. Retry logic
5. Audit report CSV
6. Short explanation of each agent responsibility

# 25. Assignment Solutions

The following solution snippets help participants after they attempt the assignments.

## Solution 1: Document-Aware Assistant

Enhance LangChain memory assistant with document tool.

In [ ]:
def document_aware_answer(claim_type):
    required = document_checklist_tool(claim_type)
    prompt = f"""
You are an insurance support assistant.

Explain required documents for a {claim_type} claim in simple language.

Required Documents:
{required}
"""
    return call_llm(prompt)

# Uncomment after API key setup.
# print(document_aware_answer("Medical"))

## Solution 2: Exponential Backoff Retry

In [ ]:
def call_llm_with_exponential_backoff(prompt, max_retries=3):
    errors = []

    for attempt in range(1, max_retries + 1):
        try:
            return call_llm(prompt), errors
        except Exception as error:
            errors.append(f"Attempt {attempt} failed: {str(error)}")
            time.sleep(2 ** attempt)

    fallback = "AI service temporarily unavailable. Manual review required."
    return fallback, errors

## Solution 3: Premium Pending Policy Test

In [ ]:
policy_database["POLPEND1"] = {
    "policy_type": "Medical",
    "policy_status": "Active",
    "premium_status": "Pending",
    "coverage_amount": 500000,
    "customer_name": "Kiran Shah"
}

premium_pending_claim = {
    "claim_id": "MED1005",
    "policy_number": "POLPEND1",
    "customer_name": "Kiran Shah",
    "claim_type": "Medical",
    "claim_amount": 90000,
    "description": "Hospitalization claim with all documents.",
    "submitted_documents": ["Hospital Bill", "Discharge Summary", "Doctor Prescription", "Claim Form"],
    "priority": "Normal"
}

final_state = claim_settlement_app.invoke({
    "claim": premium_pending_claim,
    "logs": [],
    "errors": []
})

print("Policy Status:", final_state.get("policy_status"))
print("Approval Status:", final_state.get("approval_status"))
print("Final Message:", final_state.get("final_message"))

## Solution 4: Custom Human Review Rule

In [ ]:
def custom_human_review_node(state: SettlementState) -> SettlementState:
    claim = state["claim"]
    claim_type = claim["claim_type"]
    amount = claim["claim_amount"]
    missing_docs = state.get("missing_documents", [])

    if claim_type == "Medical" and amount > 700000:
        decision = "escalate"
    elif claim_type == "Motor" and "Police Report" in missing_docs:
        decision = "rework"
    elif claim_type == "Travel" and amount < 25000 and not missing_docs:
        decision = "approve"
    elif state.get("risk_level") == "High":
        decision = "escalate"
    else:
        decision = "approve"

    logs = state.get("logs", [])
    logs.append(f"Custom Human Review completed with decision: {decision}")

    return {
        **state,
        "human_decision": decision,
        "logs": logs
    }

print("Custom human review node created.")

## Solution 5: Add Property Claim Policy and Test

In [ ]:
policy_database["PROP5001"] = {
    "policy_type": "Property",
    "policy_status": "Active",
    "premium_status": "Paid",
    "coverage_amount": 600000,
    "customer_name": "Neha Kulkarni"
}

required_documents["Property"] = [
    "Damage Photos",
    "Repair Estimate",
    "Ownership Proof",
    "Claim Form"
]

property_claim = {
    "claim_id": "PROP1001",
    "policy_number": "PROP5001",
    "customer_name": "Neha Kulkarni",
    "claim_type": "Property",
    "claim_amount": 250000,
    "description": "Water leakage damaged home interior and furniture.",
    "submitted_documents": ["Damage Photos", "Repair Estimate"],
    "priority": "Normal"
}

final_state = claim_settlement_app.invoke({
    "claim": property_claim,
    "logs": [],
    "errors": []
})

print("Approval Status:", final_state.get("approval_status"))
print("Missing Documents:", final_state.get("missing_documents"))
print("Final Message:", final_state.get("final_message"))

# 26. Review and Discussion Questions

1. Where should human review be mandatory in insurance workflows?
2. Why is LangGraph better than a single long prompt for claim settlement?
3. What is the role of memory in LangChain assistants?
4. What should be logged for audit and compliance?
5. What is the difference between rework and rejection?
6. Why should fraud risk language remain neutral?
7. How can UiPath trigger this workflow?
8. How can ServiceNow, CRM or Action Center be used for human review?
9. What should happen if the LLM API fails?
10. How can this workflow be extended with MCP tools?

# 27. Day 6 Summary

You have completed:

- LangChain vs LangGraph comparison
- LangChain with tools, memory and LLM
- Insurance-domain multi-agent LangChain use case
- LangGraph state and LLM model usage
- Human review checkpoint
- Approval, rejection, escalation and rework routing
- Retry and error recovery strategy
- Medical, motor and travel claim routing
- Six insurance practice examples
- Batch processing and audit report
- Final multi-agent claim settlement workflow assignment
- Solution examples for practice tasks

## Next Step

This workflow can be extended with:

- Real policy APIs
- UiPath Action Center
- ServiceNow approval tasks
- MCP tools
- RAG policy lookup
- LangSmith tracing
- Production monitoring